In [ ]:
"""
Plot results from eval_my_IPSLCM7_simu

"""

In [21]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
%matplotlib qt5

QStandardPaths: error creating runtime directory '/run/user/2784' (Permission denied)


In [28]:
outputpath = '/data/cburgard/EVAL_IPSLCM/'
obspath = '/data/cburgard/FOR_JACQUEMINE/'
inputpath_raw2 = '/data/cburgard/PREPARE_FORCING/PREPARE_CAVITY_MASKS/raw/'
plot_path = '/data/cburgard/PLOTS/EVAL_IPSLCM7/'

In [17]:

rrun = 'opencav-presc02'
run_name_list = [rrun]

In [6]:
domain_cfg = xr.open_dataset(inputpath_raw2 + 'eORCA1.4.3_OpenSeas_OpenAllCav_ModStraights/eORCA1.4.3_OpenSeas_OpenAllCav_ModStraights_domain_cfg.nc')

In [7]:
ds_cavity = xr.open_dataset(outputpath + rrun + '_metrics_open_cavities.nc')
ds_noncavity = xr.open_dataset(outputpath + rrun + 'metrics_outside_cavities.nc')

In [8]:
sal_obs = xr.open_dataset(obspath + 'climatology_SO_SA_10dbar_23Aug_allyear.nc')
temp_obs = xr.open_dataset(obspath + 'climatology_SO_CT_10dbar_23Aug_allyear.nc')

In [11]:
var_list_in = ['wed_gyre','ross_gyre','ACC',
            'Sbot_WWED','Sbot_EWED','Sbot_WROSS','Sbot_EROSS','Sbot_AMU','Sbot_PRYDZ',
            'Tbot_WWED','Tbot_EWED','Tbot_WROSS','Tbot_EROSS','Tbot_AMU','Tbot_PRYDZ']

In [10]:
def var_obs_mean_std(var_list_in):

    var_list = ['wed_gyre','ross_gyre','ACC',
                'Sbot_WWED','Sbot_EWED','Sbot_WROSS','Sbot_EROSS','Sbot_AMU','Sbot_PRYDZ',
                'Tbot_WWED','Tbot_EWED','Tbot_WROSS','Tbot_EROSS','Tbot_AMU','Tbot_PRYDZ']
    
    var_obs_mean = xr.DataArray(data=np.array([
                56.0, 20.0, 136.7, 
               34.9, np.nan, 35.0, np.nan, np.nan, np.nan,
               np.nan, -1.95, -1.9, -1.7, np.nan, np.nan,
    ]), dims='var').assign_coords({'var': var_list})

    var_obs_std = xr.DataArray(data=np.array([
                  8.0, 5.0, 7.8, 
                  0.0, np.nan, 0.0, np.nan, np.nan, np.nan,
                  np.nan, 0.2, 0.4, 0.4, np.nan, np.nan]), dims='var').assign_coords({'var': var_list})

    return var_obs_mean.sel(var=var_list_in), var_obs_std.sel(var=var_list_in)

In [36]:
var_obs_mean, var_obs_std = var_obs_mean_std(var_list_in)

var_to_plot = ds_noncavity

f = plt.figure()
f.set_size_inches(8.25*2, 8.25*1.5)

ax={}

leg_hdl = []

i = 0

for vv in var_list_in:
    
    ax[i] = f.add_subplot(5,3,i+1)

    if vv in list(var_to_plot.keys()):
        ax[i].plot(var_to_plot[vv], color='deepskyblue')
    
    ax[i].axhline(y=var_obs_mean.sel(var=vv), color='black', linewidth=2)
    ax[i].fill_between(x=np.arange(0,40),y1=var_obs_mean.sel(var=vv)-var_obs_std.sel(var=vv), y2=var_obs_mean.sel(var=vv)+var_obs_std.sel(var=vv), color='grey',alpha=0.2)

    if vv[0:3] == 'OHC':
        ax[i].set_title(vv+' x 10$^{22}$ J')
    else:
        ax[i].set_title(vv)

    i = i+1
#f.legend()
#f.subplots_adjust(bottom=0.05, wspace=0.1)

f.tight_layout()
sns.despine()



In [37]:
f.savefig(plot_path + 'opencav_presc02_noncavity_metrics.pdf')